# Fine-tune DAN on RAF-DB + Webcam Data

**Setup:** tạo folder `fer_finetune` trên Google Drive, upload:
| File | Source |
|---|---|
| `webcam_finetune_data.zip` | tools/prepare_finetune_data.py |
| `best_dan_model.pth` | outputs/models/ |
| `ir50.pth` | pretrain/ |
| `mobilefacenet_model_best.pth.tar` | pretrain/ |
| `RAF-DB.zip` | data/DATASET/ (zip toàn bộ) |

Sau đó sửa `DRIVE_FOLDER` ở cell dưới.

In [ ]:
DRIVE_FOLDER = '/content/drive/MyDrive/fer_finetune/colab_upload'
DRIVE_WEBZIP    = f'{DRIVE_FOLDER}/webcam_finetune_data.zip'
DRIVE_DANPT     = f'{DRIVE_FOLDER}/best_dan_model.pth'
DRIVE_RAFDB     = f'{DRIVE_FOLDER}/RAF-DB.zip'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, torch, time, zipfile
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, ConcatDataset
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
# Check files
for fp in [DRIVE_WEBZIP, DRIVE_DANPT, DRIVE_RAFDB]:
    ok = os.path.exists(fp)
    print(f'  {"[OK]" if ok else "[MISS]"} {os.path.basename(fp)}')
    if not ok:
        raise FileNotFoundError(f'Upload {os.path.basename(fp)} to {DRIVE_FOLDER}/')

In [ ]:
# Extract data
!unzip -q "{DRIVE_WEBZIP}" -d /content/webcam_data
!unzip -q "{DRIVE_RAFDB}" -d /content

for cid in range(1, 8):
    n = len(os.listdir(f'/content/webcam_data/train/{cid}'))
    print(f'  Webcam class {cid}: {n}')
print(f'  Total: {sum(len(os.listdir(f"/content/webcam_data/train/{cid}")) for cid in range(1,8))}')

In [ ]:
# DAN model
class DAN(nn.Module):
    def __init__(self, num_class=7, num_head=4):
        super().__init__()
        resnet = models.resnet18(weights=None)
        self.features = nn.Sequential(*list(resnet.children())[:-2])
        self.num_head = num_head
        self.conv_att = nn.Conv2d(512, self.num_head, kernel_size=1)
        self.fc = nn.Linear(512, num_class)
        self.bn = nn.BatchNorm1d(num_class)
    def forward(self, x):
        x = self.features(x)
        att_map = self.conv_att(x)
        att_map = att_map.view(att_map.size(0), self.num_head, -1)
        att_map = F.softmax(att_map, dim=2)
        att_map = att_map.view(att_map.size(0), self.num_head, x.size(2), x.size(3))
        x_flat = x.view(x.size(0), 1, x.size(1), -1)
        att_flat = att_map.view(att_map.size(0), self.num_head, 1, -1)
        weighted_features = (x_flat * att_flat).sum(dim=-1)
        final_features = weighted_features.mean(dim=1)
        out = self.fc(final_features)
        return self.bn(out)

In [ ]:
model = DAN(num_class=7, num_head=4).to(device)
ckpt = torch.load(DRIVE_DANPT, map_location=device)
if any(k.startswith('module.') for k in ckpt):
    ckpt = {k.replace('module.', ''): v for k, v in ckpt.items()}
model.load_state_dict(ckpt)
print('DAN weights loaded!')

In [ ]:
BATCH_SIZE = 32

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Find RAF-DB path after unzip
raf_root = '/content'
for candidate in ['/content/DATASET', '/content/train']:
    if os.path.exists(candidate):
        raf_root = os.path.dirname(candidate) if 'train' in candidate else candidate
        break
print(f'RAF-DB root: {raf_root}')

raf_train = datasets.ImageFolder(root=f'{raf_root}/train', transform=transform_train)
raf_test  = datasets.ImageFolder(root=f'{raf_root}/test', transform=transform_test)
print(f'RAF-DB: {len(raf_train)} train / {len(raf_test)} test')

webcam_train = datasets.ImageFolder(root='/content/webcam_data/train',
    transform=transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(), transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
)
print(f'Webcam: {len(webcam_train)}')

combined_train = ConcatDataset([raf_train] + [webcam_train] * 3)
train_loader = DataLoader(combined_train, BATCH_SIZE, shuffle=True, num_workers=0)
test_loader  = DataLoader(raf_test, BATCH_SIZE, shuffle=False, num_workers=0)
print(f'Train: {len(combined_train)} samples ({len(train_loader)} batches)')

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=2e-5, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

EPOCHS = 20
best_acc = 0.0
print(f'Training: {EPOCHS} epochs, lr=2e-5, batch={BATCH_SIZE}')

for epoch in range(EPOCHS):
    t0 = time.time()
    model.train()
    loss_sum, corr, cnt = 0.0, 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward(); optimizer.step()
        loss_sum += loss.item() * x.size(0)
        corr += (out.argmax(1) == y).sum().item()
        cnt += x.size(0)

    model.eval()
    vloss, vcorr, vcnt = 0.0, 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            vloss += criterion(out, y).item() * x.size(0)
            vcorr += (out.argmax(1) == y).sum().item()
            vcnt += x.size(0)

    val_acc = vcorr / vcnt
    scheduler.step(val_acc)
    print(f'Epoch {epoch+1:02d}/{EPOCHS} | {time.time()-t0:.0f}s | '
          f'Train: {loss_sum/cnt:.4f}/{corr/cnt:.4f} | Val: {vloss/vcnt:.4f}/{val_acc:.4f}')

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'best_dan_model_finetuned.pth')
        print(f'  -> Saved (val_acc={best_acc:.4f})')

print(f'Done! Best: {best_acc:.4f}')

In [ ]:
# Save to Drive + download
!cp best_dan_model_finetuned.pth "{DRIVE_FOLDER}/"
print('Copied to Drive!')
from google.colab import files
files.download('best_dan_model_finetuned.pth')